##PRE CONFIGURAÇOES QUE SERÃO REUTILIZAVEIS

In [0]:
import json
import time
import uuid
import requests

from functools import reduce
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, TimestampType, ArrayType)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

INGESTION_ID = str(uuid.uuid4())
COLLECTED_AT = datetime.now(timezone.utc)

In [0]:
def profile_table(table_name):
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.table(full_name)

    total_rows = df.count()
    profiles = []

    for field in df.schema.fields:
        column = field.name

        result = (df.agg(F.count(F.col(column)).alias("non_null_count"),
                         F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias("null_count"),
                         F.countDistinct(column).alias("distinct_count"),
                         F.min(column).cast("string").alias("minimum_value"),
                         F.max(column).cast("string").alias("maximum_value"))
                         .withColumn("table_name", F.lit(table_name))
                         .withColumn("column_name", F.lit(column))
                         .withColumn("data_type", F.lit(str(field.dataType)))
                         .withColumn("total_rows", F.lit(total_rows))
                         .withColumn("null_percentage",F.round(F.col("null_count") / F.lit(total_rows) * 100, 4))
                         .select("table_name",
                                "column_name",
                                "data_type",
                                "total_rows",
                                "non_null_count",
                                "null_count",
                                "null_percentage",
                                "distinct_count",
                                "minimum_value",
                                "maximum_value"))

        profiles.append(result)

    return reduce(lambda left, right: left.unionByName(right), profiles)

In [0]:
def create_http_session():
    retry_strategy = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session = requests.Session()
    session.mount("https://", adapter)

    return session

In [0]:
http = create_http_session()

In [0]:
def parse_partial_date(column):
    return (F.when(F.length(column) == 4, F.to_date(F.concat(column, F.lit("-01-01"))))
             .when(F.length(column) == 7, F.to_date(F.concat(column, F.lit("-01"))))
             .when(F.length(column) == 10, F.to_date(column)))

In [0]:
def save_gold(df, table_name):
    (df.write.format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}"))

In [0]:
CATALOG = "mvp_eng_dados"
SCHEMA = "mvp_cancer"
VOLUME = "mvp_repository"

In [0]:
BRONZE_STUDIES = f"{CATALOG}.{SCHEMA}.brz_clinical_trials"
bronze = spark.table(BRONZE_STUDIES)

##Schema da brz_clinical_trials pre configurado para camada silver

In [0]:
#Schema da brz_clinical_trials pre configurado para camada silver
clinical_json_schema = StructType([
    StructField("protocolSection", StructType([
        StructField("identificationModule", StructType([
            StructField("nctId", StringType()),
            StructField("briefTitle", StringType())])),

        StructField("statusModule", StructType([
            StructField("overallStatus", StringType()),
            StructField("startDateStruct", StructType([
                StructField("date", StringType()),
                StructField("type", StringType())])),
            
            StructField("completionDateStruct", StructType([
                StructField("date", StringType()),
                StructField("type", StringType())]))])),

        StructField("designModule", StructType([
            StructField("studyType", StringType()),
            StructField("phases", ArrayType(StringType())),
            StructField("enrollmentInfo", StructType([
                StructField("count", LongType()),
                StructField("type", StringType())]))])),

        StructField("sponsorCollaboratorsModule", StructType([
            StructField("leadSponsor", StructType([
                StructField("name", StringType()),
                StructField("class", StringType())]))])),

        StructField("contactsLocationsModule", StructType([
            StructField("locations", ArrayType(StructType([
                StructField("facility", StringType()),
                StructField("city", StringType()),
                StructField("state", StringType()),
                StructField("country", StringType()),
                StructField("geoPoint", StructType([
                    StructField("lat", DoubleType()),
                    StructField("lon", DoubleType())]))])))])),

        StructField("armsInterventionsModule", StructType([
            StructField("interventions", ArrayType(StructType([
                StructField("type", StringType()),
                StructField("name", StringType()),
                StructField("description", StringType())])))])),

        StructField("conditionsModule", StructType([
            StructField("conditions", ArrayType(StringType()))]))]))])

##Ler e deduplicar a Bronze

In [0]:
latest_window = (Window.partitionBy("nct_id").orderBy(F.col("collected_at").desc()))
clinical_parsed = (bronze.withColumn("row_number",F.row_number().over(latest_window)).filter(F.col("row_number") == 1).drop("row_number")
                         .withColumn("json_data",F.from_json("payload", clinical_json_schema)))

In [0]:
p = F.col("json_data.protocolSection")